In [1]:
# 1) Imports and paths
from pathlib import Path
import pandas as pd
from lexos.dtm import DTM, Vectorizer
from sklearn.svm import SVC
from lexos.classification import trainer
import lexos.corpus.corpus_stats as cs

BASE = Path.cwd()  # run this notebook from doc_src/docs/tutorials/classification
DATA = BASE / "fed_papers"

print("Base:", BASE)
print("Data dir exists:", DATA.exists())

Base: c:\Users\gabal\LocalFiles\Lexos_Independant_Research\lexos\doc_src\docs\tutorials\classification
Data dir exists: True


In [2]:
# 2) Collect train and test files
train_dirs = ["HAMILTON", "MADISON"]
test_dirs = ["COAUTHORED", "DISPUTED"]

train_files = []
for d in train_dirs:
    train_files.extend(sorted((DATA / d).glob("*.txt")))

test_files = []
for d in test_dirs:
    test_files.extend(sorted((DATA / d).glob("*.txt")))

print("Train files:", len(train_files), "Test files:", len(test_files))
assert train_files and test_files, "No files found; check working directory and folder structure."

Train files: 65 Test files: 15


In [3]:
# 3) Scrubber + Tokenizer
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer

scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")  # or any installed model

In [4]:
# 4) Build docs for CorpusStats (train + test)
from lexos.corpus.corpus_stats import CorpusStats

docs = []
raw_texts = []
sets_list = []
all_files = [*train_files, *test_files]

for f in all_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    spacy_doc = tokenizer(clean)
    tokens = [t.text for t in spacy_doc if t.text.strip()]
    doc_id = f.name
    # label inside stats is doc_id (stats only)
    docs.append((doc_id, doc_id, tokens))
    raw_texts.append(raw)          # enables char_count if CorpusStats uses raw_texts
    sets_list.append(f.parent.name)

stats = CorpusStats(docs=docs, raw_texts=raw_texts)
stats_df = stats.doc_stats_df.copy()
stats_df.insert(0, "set", sets_list)

display(stats_df.head())
print("Available columns:", list(stats_df.columns))

,set,hapax_legomena,total_tokens,total_terms,vocabulary_density,hapax_dislegomena
Documents,,,,,,
FED_11_H.txt,HAMILTON,576,2503,835,33.36,106
FED_12_H.txt,HAMILTON,555,2166,787,36.33,106
FED_13_H.txt,HAMILTON,267,967,387,40.02,46
FED_15_H.txt,HAMILTON,752,3090,1044,33.79,132
FED_16_H.txt,HAMILTON,517,2042,722,35.36,93


Available columns: ['set', 'hapax_legomena', 'total_tokens', 'total_terms', 'vocabulary_density', 'hapax_dislegomena']


In [6]:
print(stats_df.columns)

Index(['set', 'hapax_legomena', 'total_tokens', 'total_terms',
       'vocabulary_density', 'hapax_dislegomena'],
      dtype='object')


In [15]:
# 5) Select feature columns from stats_df
feature_cols = [
    'hapax_legomena', 'total_tokens', 'total_terms',
       'vocabulary_density', 'hapax_dislegomena',
]

train_mask = stats_df["set"].isin(["HAMILTON", "MADISON"])
test_mask  = stats_df["set"].isin(["COAUTHORED", "DISPUTED"])

X_train = stats_df.loc[train_mask, feature_cols].values
y_train = stats_df.loc[train_mask, "set"].values

X_test = stats_df.loc[test_mask, feature_cols].values
test_doc_ids = stats_df.loc[test_mask].index
test_sets = stats_df.loc[test_mask, "set"].values

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (65, 5) Test: (15, 5)


In [16]:
# 6) Train lexos SVC via trainer
from lexos.classification import trainer

clf = trainer.fit_classifier(
    X_train,
    y_train,
    model="naive_bayes",
)

print("Train accuracy:", clf.score(X_train, y_train))

Train accuracy: 0.8


In [19]:
y_pred_proba = clf.predict_proba(X_test)
print("Predicted probabilities for each class:")
print(y_pred_proba)

Predicted probabilities for each class:
[[1.00000000e+00 5.44977915e-12]
 [1.00000000e+00 9.44732584e-14]
 [1.00000000e+00 5.35062686e-17]
 [9.99999622e-01 3.78145341e-07]
 [9.99999999e-01 8.37156161e-10]
 [5.32097717e-01 4.67902283e-01]
 [8.26085205e-01 1.73914795e-01]
 [3.47856251e-03 9.96521437e-01]
 [1.52691311e-04 9.99847309e-01]
 [9.83369600e-01 1.66304001e-02]
 [4.98371381e-01 5.01628619e-01]
 [9.47690367e-01 5.23096330e-02]
 [9.99119648e-01 8.80351638e-04]
 [9.97069290e-01 2.93070956e-03]
 [4.80249885e-06 9.99995198e-01]]


In [20]:
print("Log probabilities of each class:")
print(clf.class_log_prior_)

Log probabilities of each class:
[-0.24256164 -1.53532994]


In [21]:
import numpy as np
feature_probs = np.exp(clf.feature_log_prob_)
print("Feature probabilities given each class:")
print(feature_probs)

Feature probabilities given each class:
[[0.13468016 0.62819229 0.19965138 0.00935571 0.02812046]
 [0.12105893 0.65515037 0.1884206  0.00684501 0.02852509]]


In [22]:
feature_names = feature_cols  # List of feature names
for i, class_label in enumerate(clf.classes_):
    print(f"Top features for class {class_label}:")
    top_features = np.argsort(clf.feature_log_prob_[i])[::-1]  # Sort in descending order
    for j in top_features[:10]:  # Top 10 features
        print(f"{feature_names[j]}: {np.exp(clf.feature_log_prob_[i][j])}")

Top features for class HAMILTON:
total_tokens: 0.6281922936814817
total_terms: 0.19965137746605208
hapax_legomena: 0.13468015714760997
hapax_dislegomena: 0.028120460557203266
vocabulary_density: 0.009355711147652324
Top features for class MADISON:
total_tokens: 0.6551503708598612
total_terms: 0.18842059966376457
hapax_legomena: 0.12105892949871669
hapax_dislegomena: 0.028525089442077572
vocabulary_density: 0.00684501053558088


In [23]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_test = clf.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(test_sets, y_pred_test))

print("\nClassification Report:")
print(classification_report(test_sets, y_pred_test))

Confusion Matrix:
[[0 0 3 0]
 [0 0 8 4]
 [0 0 0 0]
 [0 0 0 0]]

Classification Report:
              precision    recall  f1-score   support

  COAUTHORED       0.00      0.00      0.00       3.0
    DISPUTED       0.00      0.00      0.00      12.0
    HAMILTON       0.00      0.00      0.00       0.0
     MADISON       0.00      0.00      0.00       0.0

    accuracy                           0.00      15.0
   macro avg       0.00      0.00      0.00      15.0
weighted avg       0.00      0.00      0.00      15.0



c:\Users\gabal\LocalFiles\Lexos_Independant_Research\lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\gabal\LocalFiles\Lexos_Independant_Research\lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\gabal\LocalFiles\Lexos_Independant_Research\lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
 

In [24]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="accuracy")
print("Cross-validation accuracy (mean ± std):", cv_scores.mean(), "±", cv_scores.std())

Cross-validation accuracy (mean ± std): 0.7692307692307693 ± 0.12871692715908858
